# Financial Fraud Detection - Exploratory Data Analysis

This notebook performs initial exploration of the PaySim financial transactions dataset to understand fraud patterns and data characteristics.

## Load Dataset

Load the PaySim transaction dataset into a Pandas DataFrame for analysis.

In [1]:
import sys

print(sys.executable)

C:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline\.venv311\Scripts\python.exe


In [4]:
from pyspark.sql import SparkSession
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"

spark = SparkSession.builder \
    .appName("FraudDetectionPipeline") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.9


In [5]:
from pathlib import Path

project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

data_dir = project_root / "data"

csv_path = data_dir / "PS_20174392719_1491204439457_log.csv"

fraud_df = spark.read.csv(
    str(csv_path),
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully")
print("CSV path:", csv_path)

Dataset loaded successfully
CSV path: C:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline\data\PS_20174392719_1491204439457_log.csv


In [6]:
print("Rows:", fraud_df.count())
print("Columns:", len(fraud_df.columns))

fraud_df.show(5)

Rows: 6362620
Columns: 11
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|           0.0|      1|             0|
|   1| PAYMENT|11668.14|C2048537720|      41554.0|      2

## Preview Dataset

Display the first few records to understand the structure and available features.

In [9]:
spark.version

'3.5.9'

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("FraudDetectionPipeline") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.9


In [12]:
from pathlib import Path

project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

data_dir = project_root / "data"
csv_path = data_dir / "PS_20174392719_1491204439457_log.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"CSV file not found: {csv_path}")

fraud_df = spark.read.csv(
    str(csv_path),
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully")
print("CSV path:", csv_path)

Dataset loaded successfully
CSV path: c:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline\data\PS_20174392719_1491204439457_log.csv


In [26]:
fraud_df.count()

6362620

In [27]:
fraud_df.show(5)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|           0.0|      1|             0|
|   1| PAYMENT|11668.14|C2048537720|      41554.0|      29885.86|M1230701703|      

# Check Dataset Shape

In [7]:
print("Rows:", fraud_df.count())
print("Columns:", len(fraud_df.columns))

Rows: 6362620
Columns: 11


# Check Schema

In [8]:
fraud_df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



## 1. Dataset Overview

This section provides a basic understanding of the financial transaction dataset.
We will analyze the number of records, available features, and data distribution
using PySpark DataFrame operations.

In [29]:
# Total records and columns

print("Total Transactions:", fraud_df.count())
print("Total Features:", len(fraud_df.columns))

Total Transactions: 6362620
Total Features: 11


In [30]:
fraud_df.show(10)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|           0.0|      1|             0|
|   1| PAYMENT|11668.14|C2048537720|      41554.0|      29885.86|M1230701703|      

## 2. Sample Transaction Records

Displaying sample transactions to understand the structure and values of the dataset.

In [19]:
fraud_df.show(10)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|           0.0|      1|             0|
|   1| PAYMENT|11668.14|C2048537720|      41554.0|      29885.86|M1230701703|      

## 3. Missing Value Analysis

Checking missing values in each column to ensure data quality before further analysis.

In [15]:
from pyspark.sql.functions import col, sum

missing_values = fraud_df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in fraud_df.columns
    ]
)

missing_values.show()

+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|step|type|amount|nameOrig|oldbalanceOrg|newbalanceOrig|nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|   0|   0|     0|       0|            0|             0|       0|             0|             0|      0|             0|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+



## 4. Transaction Type Analysis

Analyzing different transaction categories to understand customer transaction behavior.

In [16]:
transaction_type = fraud_df.groupBy("type") \
    .count() \
    .orderBy(col("count").desc())

transaction_type.show()

+--------+-------+
|    type|  count|
+--------+-------+
|CASH_OUT|2237500|
| PAYMENT|2151495|
| CASH_IN|1399284|
|TRANSFER| 532909|
|   DEBIT|  41432|
+--------+-------+



## 5. Fraud Transaction Distribution

The target variable `isFraud` indicates whether a transaction is fraudulent.
This analysis helps identify the imbalance between normal and fraudulent transactions.

In [33]:
fraud_distribution = fraud_df.groupBy("isFraud") \
    .count()

fraud_distribution.show()

+-------+-------+
|isFraud|  count|
+-------+-------+
|      0|6354407|
|      1|   8213|
+-------+-------+



## 6. Fraud Percentage

Calculating the percentage of fraudulent transactions in the complete dataset.

In [11]:
total_transactions = fraud_df.count()

fraud_count = fraud_df.filter(
    col("isFraud") == 1
).count()

fraud_percentage = (fraud_count / total_transactions) * 100

print("Total Transactions:", total_transactions)
print("Fraud Transactions:", fraud_count)
print("Fraud Percentage:", fraud_percentage)

Total Transactions: 6362620
Fraud Transactions: 8213
Fraud Percentage: 0.12908204481801522


## 7. Transaction Amount Analysis

Analyzing transaction amounts to identify patterns between fraudulent and normal transactions.

In [12]:
fraud_df.select(
    "amount"
).describe().show()

+-------+------------------+
|summary|            amount|
+-------+------------------+
|  count|           6362620|
|   mean|179861.90354913412|
| stddev| 603858.2314629498|
|    min|               0.0|
|    max|     9.244551664E7|
+-------+------------------+



## 8. Average Amount Comparison

Comparing average transaction amounts between fraudulent and legitimate transactions.

In [36]:
fraud_df.groupBy("isFraud") \
    .avg("amount") \
    .show()

+-------+------------------+
|isFraud|       avg(amount)|
+-------+------------------+
|      0| 178197.0417274114|
|      1|1467967.2991403837|
+-------+------------------+



## 9. Fraud Pattern by Transaction Type

Identifying which transaction types contain higher fraud activity.

In [14]:
fraud_by_type = fraud_df.groupBy("type","isFraud") \
    .count() \
    .orderBy("type")

fraud_by_type.show()

+--------+-------+-------+
|    type|isFraud|  count|
+--------+-------+-------+
| CASH_IN|      0|1399284|
|CASH_OUT|      1|   4116|
|CASH_OUT|      0|2233384|
|   DEBIT|      0|  41432|
| PAYMENT|      0|2151495|
|TRANSFER|      0| 528812|
|TRANSFER|      1|   4097|
+--------+-------+-------+



## 10. Feature Correlation Analysis

Checking relationships between numerical features to identify important variables
for fraud detection modeling.

In [38]:
numeric_columns = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

for column in numeric_columns:
    correlation = fraud_df.stat.corr(column,"isFraud")
    print(column, ":", correlation)

amount : 0.07668842884028321
oldbalanceOrg : 0.010154421850332844
newbalanceOrig : -0.008148161267570215
oldbalanceDest : -0.005885278228051785
newbalanceDest : 0.0005353470683179229


In [40]:
spark.conf.set("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS")

print("Parquet configuration updated")

Parquet configuration updated



## 11. Saving Processed Dataset

Saving the Spark DataFrame in Parquet format for faster future processing.

In [42]:
from pathlib import Path
import traceback

project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

output_path = project_root / "data" / "fraud_spark_processed"

print("Saving location:", output_path)

try:
    fraud_df.coalesce(1).write \
        .mode("overwrite") \
        .parquet(str(output_path))

    print("SUCCESS")

except Exception:
    traceback.print_exc()

Saving location: c:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline\data\fraud_spark_processed


Traceback (most recent call last):
  File "C:\Users\Suji\AppData\Local\Temp\ipykernel_12204\3321393816.py", line 16, in <module>
    .parquet(str(output_path))
     ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline\.venv311\Lib\site-packages\pyspark\sql\readwriter.py", line 1721, in parquet
    self._jwrite.parquet(path)
  File "c:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline\.venv311\Lib\site-packages\py4j\java_gateway.py", line 1362, in __call__
    return_value = get_return_value(
                   ^^^^^^^^^^^^^^^^^
  File "c:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline\.venv311\Lib\site-packages\pyspark\errors\exceptions\captured.py", line 179, in deco
    return f(*a, **kw)
           ^^^^^^^^^^^
  File "c:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline\.venv311\Lib\site-packages\py4j\protocol.py", line 327, in get_return_value
    raise Py4JJavaError(
py4j.protocol.Py4JJava

## Conclusion

The Spark-based EDA identified important characteristics of financial transactions,
including fraud distribution, transaction type patterns, amount behavior, and
important numerical features. These insights will be used for feature engineering
and fraud detection model development.